# 01 — Run pipeline from exported simulation CSVs

This notebook is runnable using the exported aggregated CSV files stored under `path_od_sim_data`.
No restricted HTS microdata are required.


# Paths, seed

In [9]:
from pathlib import Path
import os
import random
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from typing import Iterable
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Reproducibility (for sampling)
SEED = int(os.getenv("PROJECT_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

# Optional: Colab Drive mount (only needed if you run in Colab)
IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DEFAULT_SIM_PATH_DATA = "/content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_data/"
path_od_sim_data = Path(os.getenv("OD_SIM_PATH_DATA", DEFAULT_SIM_PATH_DATA))

DEFAULT_SIM_PATH_RESULT = "/content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/"
path_od_sim_result = Path(os.getenv("OD_SIM_PATH_RESULT", DEFAULT_SIM_PATH_RESULT))

print("SEED =", SEED)
print("path_od_sim_data =", path_od_sim_data)
print("path_od_sim_result =", path_od_sim_result)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SEED = 42
path_od_sim_data = /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_data
path_od_sim_result = /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result


# Core utilities + pipeline functions

In [10]:
KEY_OD = ["ORIGIN_SUBZONE", "DESTINATION_SUBZONE"]
FEATURES = [
    "TRIP_DISTANCE",
    "OD_TRANSIT_DEMAND",
    "ORIGIN_SUBZONE_DEMAND",
    "DESTINATION_SUBZONE_DEMAND",
    "LU_H", "LU_W", "LU_O",
]

def od_transit_share_no_time(sc_df: pd.DataFrame) -> pd.DataFrame:
    od = sc_df.groupby(KEY_OD, as_index=False)["PT_COUNT"].sum()
    tot = float(od["PT_COUNT"].sum())
    od["OD_TRANSIT_DEMAND"] = od["PT_COUNT"] / (tot if tot > 0 else 1.0)
    return od[KEY_OD + ["OD_TRANSIT_DEMAND"]]


def demand_share(df: pd.DataFrame, col: str) -> pd.DataFrame:
    g = df.groupby(col, as_index=False)["COUNT"].sum()
    tot = float(g["COUNT"].sum())
    g[f"{col}_DEMAND"] = g["COUNT"] / (tot if tot > 0 else 1.0)
    return g[[col, f"{col}_DEMAND"]]


def ensure_unique(df: pd.DataFrame, key_cols: list[str], name: str) -> pd.DataFrame:
    """
    Force uniqueness on key_cols by summing numeric columns.
    This prevents merge multiplication and guarantees output length control.
    """
    if df.duplicated(key_cols).any():
        num_cols = [c for c in df.columns if c not in key_cols and pd.api.types.is_numeric_dtype(df[c])]
        df = df.groupby(key_cols, as_index=False)[num_cols].sum()
    return df


def apply_global_ratio_clipping(
    df: pd.DataFrame,
    survey_df: pd.DataFrame,
    walk_mode: bool,
    walk_threshold: float,
    scale: float = 10.0,
) -> pd.DataFrame:
    """
    Clip RATIO using global OT/PT ratios estimated from the survey.
    This prevents extreme ratios from unstable OD cells.
    """
    n_pt_inter = len(survey_df[(survey_df["TRAVEL_MODE"] == "PT") & (survey_df["TRIP_DISTANCE"] > 0)])
    n_pt_intra = len(survey_df[(survey_df["TRAVEL_MODE"] == "PT") & (survey_df["TRIP_DISTANCE"] == 0)])
    n_ot_inter = len(survey_df[(survey_df["TRAVEL_MODE"] == "OT") & (survey_df["TRIP_DISTANCE"] > 0)])
    n_ot_intra = len(survey_df[(survey_df["TRAVEL_MODE"] == "OT") & (survey_df["TRIP_DISTANCE"] == 0)])

    global_ratio_intra = n_ot_intra / n_pt_intra
    global_ratio_inter = n_ot_inter / n_pt_inter

    max_ratio_intra = scale * global_ratio_intra
    max_ratio_inter = scale * global_ratio_inter

    df.loc[df["TRIP_DISTANCE"] == 0, "RATIO"] = np.clip(df["RATIO"], 0.1, max_ratio_intra)
    df.loc[df["TRIP_DISTANCE"] > 0, "RATIO"] = np.clip(df["RATIO"], 0.1, max_ratio_inter)

    if walk_mode:
        df.loc[df["TRIP_DISTANCE"] >= walk_threshold, "RATIO"] = 0.0

    return df


def add_mode_ratio_with_smooth_no_time(
    survey_df: pd.DataFrame,
    lambdaa: float = 5,
    walk_mode: bool = False,
    walk_threshold: float = 3.5,
    eps: float = 1e-9,
) -> pd.DataFrame:
    """
    Smoothed OT/PT ratio per (ORIGIN_SUBZONE, DESTINATION_SUBZONE), no time dimension.
    Expects TRAVEL_MODE in {'PT','OT'}.
    """

    att_subzone = KEY_OD

    pt = survey_df.loc[survey_df["TRAVEL_MODE"] == "PT", att_subzone]
    pt = pt.assign(PT_COUNT=1)
    pt["PT_COUNT"] = pt.groupby(att_subzone)["PT_COUNT"].transform("sum")
    pt = pt.drop_duplicates(subset=att_subzone)[att_subzone + ["PT_COUNT"]]

    ot = survey_df.loc[survey_df["TRAVEL_MODE"] == "OT", att_subzone]
    ot = ot.assign(OT_COUNT=1)
    ot["OT_COUNT"] = ot.groupby(att_subzone)["OT_COUNT"].transform("sum")
    ot = ot.drop_duplicates(subset=att_subzone)[att_subzone + ["OT_COUNT"]]

    df = survey_df.merge(pt, on=att_subzone, how="left").merge(ot, on=att_subzone, how="left") #
    df["PT_COUNT"] = df["PT_COUNT"].fillna(0.0)
    df["OT_COUNT"] = df["OT_COUNT"].fillna(0.0)

    df["COUNT"] = df["PT_COUNT"] + df["OT_COUNT"]
    df["RATIO"] = df["OT_COUNT"] / (df["PT_COUNT"] + eps)

    o_pivot = (
        survey_df.groupby(["ORIGIN_SUBZONE", "TRAVEL_MODE"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={"PT": "PT_O", "OT": "OT_O"})
        .reset_index()
    )
    if "PT_O" not in o_pivot.columns:
        o_pivot["PT_O"] = 0
    if "OT_O" not in o_pivot.columns:
        o_pivot["OT_O"] = 0
    o_pivot["mu_O"] = o_pivot["OT_O"] / (o_pivot["PT_O"] + eps)

    d_pivot = (
        survey_df.groupby(["DESTINATION_SUBZONE", "TRAVEL_MODE"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={"PT": "PT_D", "OT": "OT_D"})
        .reset_index()
    )
    if "PT_D" not in d_pivot.columns:
        d_pivot["PT_D"] = 0
    if "OT_D" not in d_pivot.columns:
        d_pivot["OT_D"] = 0
    d_pivot["mu_D"] = d_pivot["OT_D"] / (d_pivot["PT_D"] + eps)

    g = survey_df["TRAVEL_MODE"].value_counts()
    mu_G = float(g.get("OT", 0.0)) / (float(g.get("PT", 0.0)) + eps)

    df = df.merge(o_pivot[["ORIGIN_SUBZONE", "mu_O", "PT_O"]], on="ORIGIN_SUBZONE", how="left")
    df = df.merge(d_pivot[["DESTINATION_SUBZONE", "mu_D", "PT_D"]], on="DESTINATION_SUBZONE", how="left")
    df["mu_G"] = mu_G

    w_o = df["PT_O"].fillna(0.0)
    w_d = df["PT_D"].fillna(0.0)
    w_sum = w_o + w_d

    mu_blend = (w_o * df["mu_O"] + w_d * df["mu_D"]) / w_sum.replace(0, np.nan)
    mu = mu_blend.fillna(df["mu_O"]).fillna(df["mu_D"]).fillna(df["mu_G"]).fillna(0.0)

    scale_val = df["COUNT"].quantile(0.98)
    if not np.isfinite(scale_val) or scale_val <= 0:
        scale_val = 1.0

    df["ww2"] = (df["COUNT"] / scale_val) ** float(lambdaa)
    df["ww2"] = df["ww2"].clip(0.0, 1.0)

    df["RATIO"] = df["ww2"] * df["RATIO"] + (1.0 - df["ww2"]) * mu
    df = df.drop(columns=["ww2"])

    df = apply_global_ratio_clipping(df, survey_df, walk_mode, walk_threshold)

    df["OT_COUNT"] = df["PT_COUNT"] * df["RATIO"]
    df = df.drop(columns=["mu_O", "mu_D", "mu_G", "PT_O", "PT_D"], errors="ignore")
    return df


def predict_ratio_to_smartcard_xgb(
    survey_feat: pd.DataFrame,
    smartcard_feat: pd.DataFrame,
    feature_cols: list[str],
) -> pd.DataFrame:
    scaler = StandardScaler()
    X_train = scaler.fit_transform(survey_feat[feature_cols])
    y_train = survey_feat["RATIO"].to_numpy(float)

    model = XGBRegressor(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
    )
    model.fit(X_train, y_train)

    X_pred = scaler.transform(smartcard_feat[feature_cols])
    ratio = model.predict(X_pred)
    ratio = np.maximum(ratio, 0.0)

    out = smartcard_feat.copy()
    out["RATIO"] = ratio
    out["OT_COUNT"] = out["RATIO"] * out["PT_COUNT"]
    return out


def fit_to_marginals_2way(
    df: pd.DataFrame,
    origin_totals,
    destination_totals,
    origin_col: str = "ORIGIN_SUBZONE",
    dest_col: str = "DESTINATION_SUBZONE",
    flow_col: str = "COUNT",
    max_iter: int = 1000,
    tol: float = 1e-10,
) -> pd.DataFrame:
    O_map = origin_totals.to_dict() if isinstance(origin_totals, pd.Series) else dict(origin_totals)
    D_map = destination_totals.to_dict() if isinstance(destination_totals, pd.Series) else dict(destination_totals)

    T_o = float(np.sum(list(O_map.values())))
    T_d = float(np.sum(list(D_map.values())))
    if T_o <= 0 or T_d <= 0:
        raise ValueError("Origin and destination marginals must have positive totals.")
    target_total = 0.5 * (T_o + T_d)

    out = df.copy()
    out["_orig_order__"] = np.arange(len(out))
    key_cols = [origin_col, dest_col]

    grp_sum = out.groupby(key_cols, as_index=False)[flow_col].sum().rename(columns={flow_col: "__agg"})
    tot_per_key = out.groupby(key_cols)[flow_col].transform("sum")
    cnt_per_key = out.groupby(key_cols)[flow_col].transform("size")
    out["__w"] = np.where(tot_per_key > 0, out[flow_col] / tot_per_key, 1.0 / cnt_per_key)

    origins = grp_sum[origin_col].unique().tolist()
    dests = grp_sum[dest_col].unique().tolist()
    O, D = len(origins), len(dests)

    o2i = {o: i for i, o in enumerate(origins)}
    d2j = {d: j for j, d in enumerate(dests)}

    X = np.zeros((O, D), dtype=float)
    for _, r in grp_sum.iterrows():
        X[o2i[r[origin_col]], d2j[r[dest_col]]] = r["__agg"]

    seed_total = X.sum()
    if seed_total <= 0:
        raise ValueError("Seed has zero total.")
    X *= (target_total / seed_total)

    O_vec = np.array([O_map.get(o, 0.0) for o in origins], dtype=float)
    D_vec = np.array([D_map.get(d, 0.0) for d in dests], dtype=float)

    if O_vec.sum() > 0:
        O_vec *= (target_total / O_vec.sum())
    if D_vec.sum() > 0:
        D_vec *= (target_total / D_vec.sum())

    Y = X.copy()
    eps_ = 1e-16
    for _ in range(max_iter):
        row_o = Y.sum(axis=1)
        Y *= np.divide(O_vec, np.maximum(row_o, eps_))[:, None]

        col_d = Y.sum(axis=0)
        Y *= np.divide(D_vec, np.maximum(col_d, eps_))[None, :]

        err_o = np.max(np.abs(Y.sum(axis=1) - O_vec))
        err_d = np.max(np.abs(Y.sum(axis=0) - D_vec))
        if max(err_o, err_d) < tol:
            break

    Y_long = pd.DataFrame(
        [(origins[i], dests[j], Y[i, j]) for i in range(O) for j in range(D) if Y[i, j] != 0.0],
        columns=[origin_col, dest_col, "__fit_od"],
    )

    out = out.merge(Y_long, on=[origin_col, dest_col], how="left")
    out[flow_col] = out["__fit_od"] * out["__w"]
    out = out.sort_values("_orig_order__").drop(columns=["_orig_order__", "__w", "__fit_od"])
    return out


def run_pipeline(
    *,
    sc_df: pd.DataFrame,                   # you pass PT as COUNT or PT_COUNT, possibly already unique
    hts_df: pd.DataFrame,
    prior_origin_vh: pd.DataFrame,
    prior_destination_vh: pd.DataFrame,
    prior_origin_wk: pd.DataFrame,
    prior_destination_wk: pd.DataFrame,
    dest_lu_mix: pd.DataFrame,
    k: float = 1.0,
    wk_threshold_km: float = 3.5,
) -> pd.DataFrame:
    """
    Guarantees output has EXACTLY the same number of rows and keys as the smartcard support.
    Strategy:
      1) Create a canonical smartcard support table sc_support unique on KEY_OD.
      2) Run VH/WK estimation.
      3) LEFT-merge estimates back onto sc_support (never OUTER-merge).
      4) Fill missing with 0.0.
    """

    # ----- 0) Canonical smartcard support (unique keys) -----
    sc = sc_df.copy()
    sc = sc.rename(columns={"COUNT": "PT_COUNT"})

    # IMPORTANT: support keys are KEY_OD only (no time)
    sc["PT_COUNT"] = sc.groupby(KEY_OD)["PT_COUNT"].transform("sum")
    sc_support = sc.drop_duplicates(subset=KEY_OD)[KEY_OD  + ['TRIP_DISTANCE', "PT_COUNT"]].reset_index(drop=True)

    # Demand share feature from smartcard
    od_pt_share = od_transit_share_no_time(sc_support)

    # ----- 1) IPF marginals -----
    O_VH = prior_origin_vh.groupby("ORIGIN_SUBZONE")["COUNT"].sum()
    D_VH = prior_destination_vh.groupby("DESTINATION_SUBZONE")["COUNT"].sum()
    total_vh = 0.5 * (float(O_VH.sum()) + float(D_VH.sum()))

    O_WK = prior_origin_wk.groupby("ORIGIN_SUBZONE")["COUNT"].sum()
    D_WK = prior_destination_wk.groupby("DESTINATION_SUBZONE")["COUNT"].sum()
    total_wk = 0.5 * (float(O_WK.sum()) + float(D_WK.sum()))

    demand_origin_vh = demand_share(prior_origin_vh, "ORIGIN_SUBZONE")
    demand_dest_vh = demand_share(prior_destination_vh, "DESTINATION_SUBZONE")
    demand_origin_wk = demand_share(prior_origin_wk, "ORIGIN_SUBZONE")
    demand_dest_wk = demand_share(prior_destination_wk, "DESTINATION_SUBZONE")

    # Ensure HTS is restricted to sc_support keys (prevents unseen OD keys)
    hts = hts_df.merge(sc_support[KEY_OD], on=KEY_OD, how="inner")

    def _stageA_for_mode(target_mode: str) -> pd.DataFrame:
        s = hts[hts["TRAVEL_MODE"].isin(["PT", target_mode])].copy()
        s.loc[s["TRAVEL_MODE"] == target_mode, "TRAVEL_MODE"] = "OT"

        s = add_mode_ratio_with_smooth_no_time(
            s,
            lambdaa=k,
            walk_mode=(target_mode == "WK"),
            walk_threshold=wk_threshold_km,
        )
        s = s[KEY_OD + ["RATIO", "TRIP_DISTANCE"]]

        if target_mode == "VH":
            dO, dD = demand_origin_vh, demand_dest_vh
        else:
            dO, dD = demand_origin_wk, demand_dest_wk

        survey_feat = (
            s.merge(dO, on="ORIGIN_SUBZONE", how="left")
             .merge(dD, on="DESTINATION_SUBZONE", how="left")
             .merge(od_pt_share, on=KEY_OD, how="left")
             .merge(dest_lu_mix, on="DESTINATION_SUBZONE", how="left")
        )


        smartcard_feat = (
            sc_support.merge(dO, on="ORIGIN_SUBZONE", how="left")
                      .merge(dD, on="DESTINATION_SUBZONE", how="left")
                      .merge(od_pt_share, on=KEY_OD, how="left")
                      .merge(dest_lu_mix, on="DESTINATION_SUBZONE", how="left")
        )


        smartcard_feat = smartcard_feat[KEY_OD + FEATURES + ["PT_COUNT"]]
        pred = predict_ratio_to_smartcard_xgb(survey_feat, smartcard_feat, FEATURES)

        if target_mode == "WK":
            pred.loc[pred["TRIP_DISTANCE"] >= wk_threshold_km, "OT_COUNT"] = 0.0

        out = pred[KEY_OD + ["OT_COUNT"]]
        out = ensure_unique(out, KEY_OD, f"{target_mode}_pred")  # defensive
        return out

    # Stage A
    vh_pred = _stageA_for_mode("VH").rename(columns={"OT_COUNT": "COUNT"})
    wk_pred = _stageA_for_mode("WK").rename(columns={"OT_COUNT": "COUNT"})

    # Renormalize to totals
    vh_pred["COUNT"] *= total_vh / vh_pred["COUNT"].sum()
    wk_pred["COUNT"] *= total_wk / wk_pred["COUNT"].sum()

    # Stage B IPF
    vh_fit = fit_to_marginals_2way(vh_pred, O_VH, D_VH, flow_col="COUNT")
    wk_fit = fit_to_marginals_2way(wk_pred, O_WK, D_WK, flow_col="COUNT")

    # Renormalize again
    vh_fit["COUNT"] *= total_vh / vh_fit["COUNT"].sum()
    wk_fit["COUNT"] *= total_wk / wk_fit["COUNT"].sum()

    vh_fit = ensure_unique(vh_fit.rename(columns={"COUNT": "VH_COUNT"}), KEY_OD, "vh_fit")[KEY_OD + ["VH_COUNT"]]
    wk_fit = ensure_unique(wk_fit.rename(columns={"COUNT": "WK_COUNT"}), KEY_OD, "wk_fit")[KEY_OD + ["WK_COUNT"]]

    # ----- 2) CRITICAL: left-merge onto sc_support to preserve length -----
    out = sc_support[KEY_OD + ["PT_COUNT"]].copy()
    out = out.merge(vh_fit, on=KEY_OD, how="left").merge(wk_fit, on=KEY_OD, how="left")
    out[["VH_COUNT", "WK_COUNT"]] = out[["VH_COUNT", "WK_COUNT"]].fillna(0.0)

    out["COUNT"] = out["PT_COUNT"] + out["VH_COUNT"] + out["WK_COUNT"]

    # Guarantee: same length and same key order as sc_support
    out = out.reset_index(drop=True)
    return out


# Run loop (loads exported CSVs, saves outputs)

In [11]:
# -----------------------------
# CONFIG
# -----------------------------
reg = "seoul"       # "sgp" or "seoul"
ver = 1           # HTS sample version: 1..5
wk_threshold_km = 3.5

k_values = [round(0.2 * i, 1) for i in range(11)] #[round(1.0,1)]
print("k_values:", k_values)

# -----------------------------
# LOAD EXPORTED INPUTS
# -----------------------------
dest_lu_mix = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_destination_lu_mix.csv")
prior_origin_vh = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_prior_origin_vh.csv")
prior_origin_wk = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_prior_origin_wk.csv")
prior_destination_vh = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_prior_destination_vh.csv")
prior_destination_wk = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_prior_destination_wk.csv")
sc_df = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_sc.csv")
hts_df = pd.read_csv(path_od_sim_data / f"data_sim_{reg}_hts_v{ver}.csv")

# Defensive: ensure required columns exist
KEY_OD = ["ORIGIN_SUBZONE", "DESTINATION_SUBZONE"]
FEATURES = ["TRIP_DISTANCE", "OD_TRANSIT_DEMAND", "ORIGIN_SUBZONE_DEMAND", "DESTINATION_SUBZONE_DEMAND",
            "LU_H", "LU_W", "LU_O"]

for k in k_values:
    result_df = run_pipeline(
        sc_df=sc_df,
        hts_df=hts_df,
        prior_origin_vh=prior_origin_vh,
        prior_destination_vh=prior_destination_vh,
        prior_origin_wk=prior_origin_wk,
        prior_destination_wk=prior_destination_wk,
        dest_lu_mix=dest_lu_mix,
        k=k,
    )
    print('Save result to', path_od_sim_result / f'result_sim_{reg}_{k}.csv')

    result_df.to_csv(path_od_sim_result / f'result_sim_{reg}_{k}.csv', index = False)


k_values: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_0.0.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_0.2.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_0.4.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_0.6.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_0.8.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_1.0.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od_matrix_generation/od_sim_result/result_sim_seoul_1.2.csv
Save result to /content/drive/MyDrive/Colab Notebooks/Workspace/od